<a href="https://colab.research.google.com/github/malleshwaramgireesh2026/Evidence_AI/blob/main/Evidence_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install numpy==1.26.4 -q

!pip install langchain==0.2.0 \
             langchain-community==0.2.0 \
             langchain-core==0.2.0 \
             langchain-groq \
             langchain-huggingface \
             langchain-text-splitters \
             sentence-transformers \
             faiss-cpu \
             pypdf \
             streamlit \
             pyngrok -q

print("All libraries installed!")
print("Now restart runtime before running next cell!")


All libraries installed!
Now restart runtime before running next cell!


In [5]:
import numpy as np
print(f"numpy: {np.__version__}")

from langchain_groq import ChatGroq
print("langchain_groq ok")

from langchain_huggingface import HuggingFaceEmbeddings
print("langchain_huggingface ok")

from langchain_community.vectorstores import FAISS
print("FAISS ok")

from langchain_community.document_loaders import PyPDFLoader
print("PyPDFLoader ok")

import streamlit
print(f"streamlit: {streamlit.__version__}")

print("\n All libraries ready!")

numpy: 1.26.4
langchain_groq ok
langchain_huggingface ok
FAISS ok
PyPDFLoader ok
streamlit: 1.57.0

 All libraries ready!


In [6]:
import os
from langchain_groq import ChatGroq

# Set your Groq API key
os.environ["GROQ_API_KEY"] = "Paste Your API Key"

# Initialize free LLaMA 3.1 via Groq
llm = ChatGroq(
    model_name="llama-3.1-8b-instant",
    temperature=0,
    groq_api_key=os.environ["GROQ_API_KEY"]
)

# Quick test
response = llm.invoke("How an API works?")
print("LLM working!")
print(f"Test response: {response.content}")
#print(response)


LLM working!
Test response: **API Overview**

An API (Application Programming Interface) is a set of defined rules that enables different software systems to communicate with each other. It acts as an intermediary between a client (the application making the request) and a server (the application providing the data or service).

**API Components**
-----------------

1. **Client**: The application making the request to the API. This can be a web application, mobile app, or any other software system.
2. **Server**: The application providing the data or service. This can be a web server, database, or any other system that provides data or services.
3. **API Gateway**: The entry point for API requests. It receives requests from clients and forwards them to the appropriate server.
4. **API Endpoints**: Specific URLs that clients can use to interact with the API. Each endpoint typically corresponds to a specific action or resource.

**API Request-Response Cycle**
----------------------------

In [7]:
GROQ_KEY = "Paste your API Key"

with open("app.py", "w", encoding="utf-8") as f:
    f.write("import os\n")
    f.write(f'os.environ["GROQ_API_KEY"] = "{GROQ_KEY}"\n')
    f.write(r'''
import os
import json
import shutil
from datetime import datetime
from pathlib import Path

import streamlit as st
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

WORKSPACE_ROOT = "/content/rag_workspaces"

st.set_page_config(
    page_title="EvidenceFlow AI",
    page_icon="🔎",
    layout="wide"
)

st.markdown("""
<style>
.main .block-container {
    max-width: 1120px;
    padding-top: 2rem;
}
.header-box {
    padding: 1.5rem 1.7rem;
    border-radius: 12px;
    background: #f6f9fc;
    border: 1px solid #dce8f5;
    margin-bottom: 1.2rem;
}
.header-title {
    font-size: 2.1rem;
    font-weight: 760;
    color: #102033;
    margin-bottom: 0.35rem;
}
.header-subtitle {
    font-size: 1rem;
    color: #5c6f82;
    max-width: 850px;
}
.section-title {
    font-size: 1.1rem;
    font-weight: 700;
    color: #102033;
    margin-top: 1rem;
    margin-bottom: 0.55rem;
}
.answer-card {
    padding: 1.15rem 1.25rem;
    border-radius: 10px;
    border: 1px solid #dce8f5;
    background: #ffffff;
    color: #102033;
    line-height: 1.65;
    font-size: 1rem;
}
.source-box {
    color: #26384d;
    line-height: 1.55;
    white-space: pre-wrap;
}
.disclaimer-box {
    padding: 0.95rem 1.05rem;
    border-radius: 10px;
    background: #fff8e6;
    border: 1px solid #f0d98c;
    color: #6b5200;
    font-size: 0.92rem;
}
.stTextArea textarea {
    border-radius: 10px;
}
div.stButton > button[kind="primary"] {
    background-color: #1f6feb;
    border-color: #1f6feb;
    color: white;
}
div.stButton > button[kind="primary"]:hover {
    background-color: #185abc;
    border-color: #185abc;
    color: white;
}
</style>
""", unsafe_allow_html=True)

Path(WORKSPACE_ROOT).mkdir(parents=True, exist_ok=True)

def clean_workspace_name(name):
    cleaned = "".join(ch if ch.isalnum() or ch in ["_", "-"] else "_" for ch in name.strip())
    return cleaned or "default_workspace"

def workspace_path(workspace_name):
    return Path(WORKSPACE_ROOT) / clean_workspace_name(workspace_name)

def docs_path(workspace_name):
    return workspace_path(workspace_name) / "documents"

def index_path(workspace_name):
    return workspace_path(workspace_name) / "faiss_index"

def metadata_path(workspace_name):
    return workspace_path(workspace_name) / "metadata.json"

def chat_path(workspace_name):
    return workspace_path(workspace_name) / "chat_history.json"

def summary_path(workspace_name):
    return workspace_path(workspace_name) / "summary.txt"

def ensure_workspace(workspace_name):
    workspace_path(workspace_name).mkdir(parents=True, exist_ok=True)
    docs_path(workspace_name).mkdir(parents=True, exist_ok=True)

def list_workspaces():
    root = Path(WORKSPACE_ROOT)
    names = [item.name for item in root.iterdir() if item.is_dir()]
    return sorted(names) if names else ["default_workspace"]

def load_json(path, default):
    try:
        if Path(path).exists():
            with open(path, "r", encoding="utf-8") as file:
                return json.load(file)
    except Exception:
        pass
    return default

def save_json(path, data):
    with open(path, "w", encoding="utf-8") as file:
        json.dump(data, file, indent=2, ensure_ascii=False)

def load_chat_history(workspace_name):
    return load_json(chat_path(workspace_name), [])

def save_chat_history(workspace_name, history):
    save_json(chat_path(workspace_name), history)

def get_item_mode(item):
    return item.get("mode", item.get("query_type", "Unknown"))

existing_workspaces = list_workspaces()

if "active_workspace" not in st.session_state:
    st.session_state.active_workspace = existing_workspaces[0]

if st.session_state.active_workspace not in existing_workspaces:
    st.session_state.active_workspace = existing_workspaces[0]

if "uploader_reset_token" not in st.session_state:
    st.session_state.uploader_reset_token = 0

if "workspace_input_token" not in st.session_state:
    st.session_state.workspace_input_token = 0

if "workspace_created_toast" not in st.session_state:
    st.session_state.workspace_created_toast = ""

if st.session_state.workspace_created_toast:
    st.toast(
        f"Workspace created: {st.session_state.workspace_created_toast}",
        icon="✅"
    )
    st.session_state.workspace_created_toast = ""

@st.cache_resource(show_spinner=False)
def get_embeddings():
    return HuggingFaceEmbeddings(
        model_name="all-MiniLM-L6-v2",
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True}
    )

@st.cache_resource(show_spinner=False)
def get_llm():
    return ChatGroq(
        model_name="llama-3.1-8b-instant",
        temperature=0,
        groq_api_key=os.environ["GROQ_API_KEY"]
    )

def save_uploaded_files(workspace_name, uploaded_files):
    ensure_workspace(workspace_name)

    for uploaded_file in uploaded_files:
        target_path = docs_path(workspace_name) / uploaded_file.name
        with open(target_path, "wb") as file:
            file.write(uploaded_file.getvalue())

def load_documents_from_workspace(workspace_name):
    all_docs = []
    document_folder = docs_path(workspace_name)

    if not document_folder.exists():
        return all_docs

    for file_path in document_folder.iterdir():
        suffix = file_path.suffix.lower()

        try:
            if suffix == ".pdf":
                loader = PyPDFLoader(str(file_path))
                docs = loader.load()
            elif suffix in [".txt", ".md"]:
                loader = TextLoader(str(file_path), encoding="utf-8")
                docs = loader.load()
            else:
                continue

            for doc in docs:
                doc.metadata["source"] = file_path.name
                doc.metadata["document_name"] = file_path.name
                if "page" not in doc.metadata:
                    doc.metadata["page"] = "N/A"

            all_docs.extend(docs)

        except Exception as error:
            st.warning(f"Could not read {file_path.name}: {error}")

    return all_docs

def build_and_save_vectorstore(workspace_name):
    raw_docs = load_documents_from_workspace(workspace_name)

    if not raw_docs:
        return None, []

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=120
    )

    chunks = splitter.split_documents(raw_docs)
    embeddings = get_embeddings()

    vectorstore = FAISS.from_documents(chunks, embeddings)

    index_path(workspace_name).mkdir(parents=True, exist_ok=True)
    vectorstore.save_local(str(index_path(workspace_name)))

    metadata = {
        "workspace": workspace_name,
        "updated_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "documents": sorted([item.name for item in docs_path(workspace_name).iterdir()]),
        "document_count": len(list(docs_path(workspace_name).iterdir())),
        "chunk_count": len(chunks)
    }

    save_json(metadata_path(workspace_name), metadata)

    return vectorstore, chunks

def load_vectorstore(workspace_name):
    if not index_path(workspace_name).exists():
        return None

    try:
        return FAISS.load_local(
            str(index_path(workspace_name)),
            get_embeddings(),
            allow_dangerous_deserialization=True
        )
    except Exception:
        return None

def format_docs(docs):
    formatted = []

    for doc in docs:
        source = doc.metadata.get("document_name", doc.metadata.get("source", "Unknown document"))
        page = doc.metadata.get("page", "N/A")

        formatted.append(
            f"[Document: {source} | Page: {page}]\n{doc.page_content}"
        )

    return "\n\n".join(formatted)

def get_recent_memory(history, limit=3):
    if not history:
        return "No previous conversation."

    recent = history[:limit]
    memory_parts = []

    for item in recent:
        memory_parts.append(
            f"Previous question: {item.get('question', '')}\nPrevious answer: {item.get('answer', '')}"
        )

    return "\n\n".join(memory_parts)

def score_to_relevance(score):
    try:
        return round((1 / (1 + float(score))) * 100, 1)
    except Exception:
        return 0.0

def retrieve_with_scores(vectorstore, question, k=7):
    results = vectorstore.similarity_search_with_score(question, k=k)
    docs = []
    scored_docs = []

    for doc, score in results:
        relevance = score_to_relevance(score)
        doc.metadata["relevance_score"] = relevance
        docs.append(doc)
        scored_docs.append((doc, relevance))

    return docs, scored_docs

def evaluate_answer(answer, docs, question):
    context_text = " ".join([doc.page_content for doc in docs]).lower()
    answer_lower = answer.lower()
    question_lower = question.lower()

    answer_words = [
        w.strip(".,;:()[]")
        for w in answer_lower.split()
        if len(w.strip(".,;:()[]")) > 4
    ]

    matched = [w for w in answer_words[:40] if w in context_text]
    grounding_score = min(len(matched) / 12, 1.0)

    has_citation = any(
        term in answer_lower
        for term in ["page", "document", "section", "table", "figure", "source"]
    )

    question_words = [
        w.strip(".,;:()[]")
        for w in question_lower.split()
        if len(w.strip(".,;:()[]")) > 4
    ]

    relevant_words = [w for w in question_words if w in answer_lower]
    relevance_score = min(len(relevant_words) / max(len(question_words), 1), 1.0)
    completeness_score = min(len(answer.split()) / 50, 1.0)

    overall_score = round(
        (
            grounding_score
            + relevance_score
            + int(has_citation)
            + completeness_score
        ) / 4 * 100,
        1
    )

    return {
        "overall_score": overall_score,
        "grounding_score": round(grounding_score * 100, 1),
        "relevance_score": round(relevance_score * 100, 1),
        "citation_score": 100.0 if has_citation else 0.0,
        "completeness_score": round(completeness_score * 100, 1),
        "has_citation": has_citation,
        "says_insufficient": "insufficient information" in answer_lower,
        "low_confidence": overall_score < 55
    }

answer_prompt = PromptTemplate(
    input_variables=["context", "question", "memory", "query_type"],
    template="""
You are EvidenceFlow AI, a professional source-grounded document analysis assistant.

Query type:
{query_type}

Conversation memory:
{memory}

Answer ONLY using the provided document context.

Rules:
- If the answer is not present in the context, say: Insufficient information in provided documents
- Mention supporting document names and page numbers when available
- Do not invent facts, numbers, dates, policies, claims, or conclusions
- If evidence is weak or conflicting, say that clearly
- Keep the answer clear, structured, and professional

Context:
{context}

Question:
{question}

Answer:
"""
)

compare_prompt = PromptTemplate(
    input_variables=["context", "question", "memory"],
    template="""
You are EvidenceFlow AI, a professional source-grounded document comparison assistant.

Conversation memory:
{memory}

Compare the uploaded documents using ONLY the provided context.

Return:
1. Direct answer
2. Where documents agree
3. Where documents differ
4. Missing or weak evidence
5. Best supported conclusion
6. Citations with document names and pages

Rules:
- Do not invent facts
- If evidence is insufficient, say so clearly
- Mention document names and page numbers when available

Context from multiple documents:
{context}

Comparison question:
{question}

Comparison report:
"""
)

summary_prompt = PromptTemplate(
    input_variables=["context"],
    template="""
You are EvidenceFlow AI.

Create a concise workspace summary using ONLY the provided document context.

Return:
1. Overview
2. Key topics
3. Important facts or recommendations
4. Useful questions to ask

Context:
{context}

Summary:
"""
)

def generate_workspace_summary(workspace_name, chunks):
    if not chunks:
        return "No documents available for summary."

    llm = get_llm()
    context = "\n\n".join([doc.page_content[:1000] for doc in chunks[:8]])
    chain = summary_prompt | llm | StrOutputParser()
    summary = chain.invoke({"context": context})

    with open(summary_path(workspace_name), "w", encoding="utf-8") as file:
        file.write(summary)

    return summary

def run_rag(vectorstore, question, query_type, history, compare=False):
    docs, scored_docs = retrieve_with_scores(vectorstore, question, k=7)
    context = format_docs(docs)
    memory = get_recent_memory(history)

    if compare:
        chain = compare_prompt | get_llm() | StrOutputParser()
        answer = chain.invoke({
            "context": context,
            "question": question,
            "memory": memory
        })
    else:
        chain = answer_prompt | get_llm() | StrOutputParser()
        answer = chain.invoke({
            "context": context,
            "question": question,
            "memory": memory,
            "query_type": query_type
        })

    evaluation = evaluate_answer(answer, docs, question)
    return answer, docs, scored_docs, evaluation

def render_sources(scored_docs):
    st.markdown('<div class="section-title">Sources</div>', unsafe_allow_html=True)

    for i, (doc, relevance) in enumerate(scored_docs, start=1):
        source = doc.metadata.get("document_name", doc.metadata.get("source", "Unknown document"))
        page = doc.metadata.get("page", "N/A")

        with st.expander(f"Source {i} | {source} | Page {page} | Relevance {relevance}%"):
            st.markdown(
                f'<div class="source-box">{doc.page_content[:1200]}...</div>',
                unsafe_allow_html=True
            )

def render_evaluation(evaluation):
    with st.expander("View answer evaluation"):
        e1, e2, e3, e4, e5 = st.columns(5)
        e1.metric("Overall", f"{evaluation['overall_score']}%")
        e2.metric("Grounding", f"{evaluation['grounding_score']}%")
        e3.metric("Relevance", f"{evaluation['relevance_score']}%")
        e4.metric("Citation", f"{evaluation['citation_score']}%")
        e5.metric("Completeness", f"{evaluation['completeness_score']}%")

        if evaluation["low_confidence"]:
            st.warning("Low-confidence answer. Review the sources before using it.")
        elif not evaluation["has_citation"]:
            st.warning("Citation may be missing. Review the sources before using it.")
        else:
            st.success("Answer appears grounded and includes source references.")

st.markdown("""
<div class="header-box">
    <div class="header-title">EvidenceFlow AI</div>
    <div class="header-subtitle">
        Source-grounded answers, document comparison, and evidence trails for uploaded documents.
    </div>
</div>
""", unsafe_allow_html=True)

with st.sidebar:
    st.markdown("## Workspace")

    existing_workspaces = list_workspaces()

    selected_workspace_choice = st.selectbox(
        "Select workspace",
        existing_workspaces,
        index=existing_workspaces.index(st.session_state.active_workspace),
        key="workspace_selector"
    )

    if selected_workspace_choice != st.session_state.active_workspace:
        st.session_state.active_workspace = selected_workspace_choice
        st.session_state.uploader_reset_token += 1
        st.rerun()

    new_workspace = st.text_input(
        "Create workspace",
        placeholder="example: research_papers",
        key=f"new_workspace_name_{st.session_state.workspace_input_token}"
    )

    if st.button("Create Workspace", use_container_width=True):
        if new_workspace.strip():
            new_workspace_name = clean_workspace_name(new_workspace)
            ensure_workspace(new_workspace_name)

            st.session_state.active_workspace = new_workspace_name
            st.session_state.workspace_input_token += 1
            st.session_state.uploader_reset_token += 1
            st.session_state.workspace_created_toast = new_workspace_name

            st.rerun()
        else:
            st.warning("Enter a workspace name first.")

    selected_workspace = st.session_state.active_workspace
    ensure_workspace(selected_workspace)

    st.divider()

    st.markdown("## Chat History")
    chat_history = load_chat_history(selected_workspace)

    if chat_history:
        history_text = ""

        for i, item in enumerate(chat_history, start=1):
            item_mode = get_item_mode(item)

            history_text += f"EvidenceFlow Chat {i}\n"
            history_text += f"Time: {item.get('time', 'N/A')}\n"
            history_text += f"Workspace: {selected_workspace}\n"
            history_text += f"Mode: {item_mode}\n"
            history_text += f"Question: {item.get('question', '')}\n"
            history_text += f"Answer:\n{item.get('answer', '')}\n"
            history_text += "-" * 70 + "\n\n"

        st.download_button(
            "Download History",
            data=history_text,
            file_name=f"{selected_workspace}_history.txt",
            mime="text/plain",
            use_container_width=True
        )

        if st.button("Clear History", use_container_width=True):
            save_chat_history(selected_workspace, [])
            st.rerun()

        for i, item in enumerate(chat_history[:5], start=1):
            item_mode = get_item_mode(item)

            with st.expander(f"{i}. {item.get('question', '')[:38]}..."):
                st.caption(f"{item.get('time', 'N/A')} | {item_mode}")
                st.write(item.get("answer", ""))

    else:
        st.caption("No chat history yet.")

metadata = load_json(metadata_path(selected_workspace), {})
chat_history = load_chat_history(selected_workspace)
vectorstore = load_vectorstore(selected_workspace)

tab_workspace, tab_ask, tab_compare = st.tabs([
    "Workspace",
    "Ask Documents",
    "Compare Documents"
])

with tab_workspace:
    st.markdown('<div class="section-title">Documents</div>', unsafe_allow_html=True)

    uploaded_files = st.file_uploader(
        "Upload documents",
        type=["pdf", "txt", "md"],
        accept_multiple_files=True,
        key=f"uploader_{selected_workspace}_{st.session_state.uploader_reset_token}"
    )

    build_col, clear_col = st.columns([0.35, 0.65])

    with build_col:
        build_button = st.button("Build Index", type="primary", use_container_width=True)

    with clear_col:
        clear_workspace = st.button("Clear Workspace", use_container_width=False)

    if build_button:
        if not uploaded_files:
            st.warning("Please upload at least one document.")
        else:
            with st.spinner("Preparing workspace. This may take a moment..."):
                save_uploaded_files(selected_workspace, uploaded_files)
                vectorstore, chunks = build_and_save_vectorstore(selected_workspace)
                generate_workspace_summary(selected_workspace, chunks)

            st.session_state.uploader_reset_token += 1
            st.success("Workspace indexed successfully.")
            st.rerun()

    if clear_workspace:
        if workspace_path(selected_workspace).exists():
            shutil.rmtree(workspace_path(selected_workspace))
        ensure_workspace(selected_workspace)
        st.session_state.uploader_reset_token += 1
        st.cache_resource.clear()
        st.success("Workspace cleared.")
        st.rerun()

    st.caption("Supported formats: PDF, TXT, MD")

    st.markdown('<div class="section-title">Workspace Status</div>', unsafe_allow_html=True)

    c1, c2, c3 = st.columns(3)
    c1.metric("Documents", metadata.get("document_count", 0))
    c2.metric("Chunks", metadata.get("chunk_count", 0))
    c3.metric("Questions", len(chat_history))

    if metadata.get("updated_at"):
        st.caption(f"Index updated: {metadata['updated_at']}")

    st.markdown('<div class="section-title">Workspace Summary</div>', unsafe_allow_html=True)

    if summary_path(selected_workspace).exists():
        with open(summary_path(selected_workspace), "r", encoding="utf-8") as file:
            st.write(file.read())
    else:
        st.caption("No summary yet. Build an index to generate one.")

with tab_ask:
    if vectorstore is None:
        st.info("Go to the Workspace tab, upload documents, and build the index first.")
    else:
        st.success(f"Workspace ready: {selected_workspace}")

        query_type = st.selectbox(
            "Question type",
            [
                "Direct Answer",
                "Summarize",
                "Extract Key Facts",
                "Find Risks or Warnings",
                "Generate Study Notes"
            ]
        )

        question = st.text_area(
            "Question",
            placeholder="Ask a question from the uploaded documents.",
            height=120,
            key="ask_question"
        )

        ask_button = st.button("Get Answer", type="primary", use_container_width=False)

        if ask_button:
            if not question.strip():
                st.warning("Please enter a question.")
            else:
                st.info("Searching documents and preparing a grounded answer...")

                with st.spinner("Working on your answer..."):
                    answer, docs, scored_docs, evaluation = run_rag(
                        vectorstore,
                        question.strip(),
                        query_type,
                        chat_history
                    )

                chat_history.insert(0, {
                    "time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                    "mode": query_type,
                    "question": question.strip(),
                    "answer": answer,
                    "evaluation": evaluation
                })
                save_chat_history(selected_workspace, chat_history)

                st.markdown('<div class="section-title">Answer</div>', unsafe_allow_html=True)
                st.markdown(f'<div class="answer-card">{answer}</div>', unsafe_allow_html=True)

                render_evaluation(evaluation)
                render_sources(scored_docs)

with tab_compare:
    if vectorstore is None:
        st.info("Go to the Workspace tab, upload documents, and build the index first.")
    else:
        st.success(f"Workspace ready: {selected_workspace}")

        compare_question = st.text_area(
            "Comparison question",
            placeholder="Example: Compare the recommendations across the uploaded documents.",
            height=120,
            key="compare_question"
        )

        compare_button = st.button("Generate Comparison Report", type="primary")

        if compare_button:
            if not compare_question.strip():
                st.warning("Please enter a comparison question.")
            else:
                st.info("Comparing documents and preparing the best supported answer...")

                with st.spinner("Building comparison report..."):
                    answer, docs, scored_docs, evaluation = run_rag(
                        vectorstore,
                        compare_question.strip(),
                        "Compare Documents",
                        chat_history,
                        compare=True
                    )

                chat_history.insert(0, {
                    "time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                    "mode": "Compare Documents",
                    "question": compare_question.strip(),
                    "answer": answer,
                    "evaluation": evaluation
                })
                save_chat_history(selected_workspace, chat_history)

                st.markdown('<div class="section-title">Comparison Report</div>', unsafe_allow_html=True)
                st.markdown(f'<div class="answer-card">{answer}</div>', unsafe_allow_html=True)

                render_evaluation(evaluation)
                render_sources(scored_docs)

st.divider()

st.markdown("""
<div class="disclaimer-box">
    <strong>EvidenceFlow AI disclaimer:</strong>
    Answers are generated only from uploaded documents and retrieved source passages.
    Verify important decisions against original source files.
</div>
""", unsafe_allow_html=True)
''')

print("EvidenceFlow AI app updated with safe workspace input reset.")


EvidenceFlow AI app updated with safe workspace input reset.


In [8]:
import os
import time
import shutil
import subprocess
from pathlib import Path
from pyngrok import ngrok

# 1. Stop old Streamlit and ngrok sessions
os.system("pkill -f streamlit")
ngrok.kill()
time.sleep(3)

# 2. Clear all old local EvidenceFlow workspaces
workspace_root = Path("/content/rag_workspaces")

if workspace_root.exists():
    shutil.rmtree(workspace_root)

workspace_root.mkdir(parents=True, exist_ok=True)

print("Fresh EvidenceFlow workspace created.")

# 3. Start Streamlit
process = subprocess.Popen(
    [
        "streamlit", "run", "app.py",
        "--server.port", "8501",
        "--server.headless", "true",
        "--server.address", "0.0.0.0"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

print("Starting Streamlit...")
time.sleep(10)

# 4. Start ngrok tunnel
ngrok.set_auth_token("Paste yuor Token")
public_url = ngrok.connect(addr="8501", proto="http")

print(f"Open your fresh EvidenceFlow AI app here: {public_url}")



Fresh EvidenceFlow workspace created.
Starting Streamlit...
Open your fresh EvidenceFlow AI app here: NgrokTunnel: "https://antitoxic-manila-enamel.ngrok-free.dev" -> "http://localhost:8501"
